# La convolution, pas à pas — sur les signaux SpectraSense

Objectif : **comprendre** ce que fait une couche de convolution avant de concevoir le réseau.
Règle du jeu, comme pour MNIST : **prédire avant d'exécuter**. Chaque étape commence par une
question ; écris ta réponse dans la cellule « Mes prédictions » avant de lancer la cellule suivante.

| Étape | Question |
|---|---|
| 1 | Une convolution, c'est quoi ? Trois filtres faits à la main, sur de vrais événements |
| 2 | Pourquoi ton perceptron a du mal : ses neurones sont attachés à des **positions** |
| 3 | Une seule couche apprise : qu'a-t-elle trouvé ? (réseau entraîné dans le banc) |
| 4 | Empiler les couches : ce qu'un neurone « voit » (champ récepteur), et le piège du pooling |
| 5 | Concevoir notre réseau à partir des motifs à reconnaître |

Lien avec Nielsen, chapitre 6 : *local receptive fields*, *shared weights*, *pooling*.
Données : `donnees/evenements_spectrasense.npz` (8 événements réels, jours d'entraînement ;
fabriqué par `spectrasense/tools/dataset/evenements_exemples.py`).

In [ ]:
import os, sys, json
import numpy as np
import matplotlib.pyplot as plt

ICI = os.path.abspath("") if os.path.basename(os.path.abspath("")) == "experiences" else os.path.join(os.path.abspath(""), "experiences")
RACINE = os.path.dirname(ICI)
sys.path.insert(0, RACINE)

z = np.load(os.path.join(ICI, "donnees", "evenements_spectrasense.npz"))
META = json.loads(str(z["meta"]))
CLASSES = json.loads(str(z["classes"]))
EV = {nom: {"p": z[nom + "__p"], "x": z[nom + "__x"], "y": z[nom + "__y"], **m} for nom, m in META.items()}
COURT = ["repos", "chauffe", "régulation", "fin allum.", "broyage", "extraction", "fin café", "pic",
         "extinction", "grille-pain", "bouilloire", "inconnu"]
COULEURS = ["#dddddd", "#d62728", "#ff9896", "#8c564b", "#1f77b4", "#17becf", "#9edae5", "#bcbd22",
            "#7f7f7f", "#ff7f0e", "#9467bd", "#2ca02c"]

def bandeau(ax, y, t):
    # classe vraie de chaque seconde, en bandeau coloré en haut du graphique
    for k in range(len(y)):
        if y[k] >= 0:
            ax.axvspan(t[k] - .5, t[k] + .5, ymin=.92, ymax=1, color=COULEURS[y[k]], lw=0)

def legende(fig):
    fig.legend([plt.Rectangle((0, 0), 1, 1, color=c) for c in COULEURS], COURT, ncol=12, fontsize=7,
               loc="lower center")

for nom, e in EV.items():
    print(f"{nom:20s} {len(e['p'])} s, début à t = 0 ({e['heure'][:16]})")

## Étape 1 — Une convolution, c'est un petit neurone qu'on fait glisser

Un **filtre** de k poids `w = [w0, w1, …, w(k-1)]` (plus un biais b) regarde k secondes consécutives :

$$y[t] = w_0\,x[t] + w_1\,x[t+1] + \dots + w_{k-1}\,x[t+k-1] + b$$

puis avance d'une seconde et recommence **avec les mêmes poids** (*poids partagés*). La sortie y
est une nouvelle courbe, plus courte de k−1 secondes (convolution « valide », comme dans le banc) :
elle dit, à chaque instant, *à quel point le motif du filtre est présent là*.

C'est exactement un neurone de Nielsen (somme pondérée + biais), mais branché sur k entrées
voisines au lieu de toutes, et recopié le long du temps.

In [ ]:
def convolution(x, w, b=0.0):
    # convolution 1D « valide », écrite à la main : la même opération que la couche conv1d du banc
    k = len(w)
    y = np.zeros(len(x) - k + 1)
    for t in range(len(y)):
        y[t] = sum(w[j] * x[t + j] for j in range(k)) + b
    return y

# contrôle : identique à la corrélation de numpy
x_essai = np.random.default_rng(0).random(20)
assert np.allclose(convolution(x_essai, [1, -2, 1]), np.correlate(x_essai, [1, -2, 1], mode="valid"))
print("ok")

### Exercice 1 — prédire

On applique trois filtres au **café n° 1** (canal linéaire, P / 3000 : 1400 W ≈ 0,47) :

| Filtre | Poids |
|---|---|
| A | `[-1, +1]` |
| B | `[1/3, 1/3, 1/3]` |
| C | `[+1, -2, +1]` |

Pour chacun : à quoi ressemblera la courbe de sortie ? Où sera-t-elle positive, négative, nulle ?
Que se passe-t-il au début du café, pendant les creux du broyage, sur le palier d'extraction, à la fin ?

**Mes prédictions :**

- A :
- B :
- C :

In [ ]:
def montrer(nom, filtres, canal=1, titre_canal="P / 3000"):
    e = EV[nom]
    x, y = e["x"][canal], e["y"]
    t = np.arange(len(x)) - e["debut"]
    fig, axes = plt.subplots(len(filtres) + 1, 1, figsize=(14, 2.1 * (len(filtres) + 1)), sharex=True)
    axes[0].plot(t, x, c="k", lw=1); bandeau(axes[0], y, t)
    axes[0].set_ylabel(titre_canal); axes[0].set_title(f"{nom} — signal d'entrée", fontsize=9)
    for ax, (lettre, w) in zip(axes[1:], filtres.items()):
        s = convolution(x, w)
        ts = t[:len(s)] + (len(w) - 1)          # sortie placée sur la DERNIÈRE seconde vue par le filtre
        ax.plot(ts, s, c="tab:blue", lw=1); ax.axhline(0, c="0.6", lw=.5)
        ax.set_ylabel(lettre); ax.set_title(f"filtre {lettre} = {np.round(w, 3).tolist()}", fontsize=9)
    axes[-1].set_xlabel("secondes depuis le début de l'événement")
    legende(fig); plt.tight_layout(rect=(0, .03, 1, 1)); plt.show()

FILTRES = {"A": [-1, 1], "B": [1/3, 1/3, 1/3], "C": [1, -2, 1]}
montrer("cafe_1", FILTRES)

### À observer

- **A** est une *dérivée* : grand positif sur une montée, grand négatif sur une descente, ~0 sur un palier.
  C'est un **détecteur de fronts**. Il ne dit rien du niveau (1400 W ou 2200 W donnent le même zéro sur un palier).
- **B** est une *moyenne* : il lisse, efface les creux d'une seconde. Il garde le niveau, perd les détails.
- **C** compare la seconde du milieu à ses voisines : fort sur un **creux** ou un **pic** isolé (signes opposés),
  ~0 sur un palier **et** sur une rampe régulière.

Un réseau de convolution, c'est des dizaines de filtres de ce genre, **dont les poids sont appris**
au lieu d'être choisis. La question de l'étape 3 sera : retrouve-t-il ces filtres-là ?

### Exercice 2 — à toi de concevoir un filtre

Invente un filtre (3 à 9 poids) qui réagit fortement à un **pic de maintien** (1 à 3 s vers 500 W,
puis retour à ~0) mais peu au **début d'un café** (montée qui reste haute).
Indice : un pic *monte puis redescend*, un café *monte et reste*. Teste-le sur tous les événements.

In [ ]:
MON_FILTRE = [-1, 1, 1, -1]          # à modifier

fig, axes = plt.subplots(4, 4, figsize=(15, 8))
for (nom, e), (a_sig, a_sor) in zip(EV.items(), axes.reshape(-1, 2)):
    x = e["x"][1]; t = np.arange(len(x)) - e["debut"]
    s = convolution(x, MON_FILTRE)
    a_sig.plot(t, x, c="k", lw=.8); bandeau(a_sig, e["y"], t); a_sig.set_title(nom, fontsize=8)
    a_sor.plot(t[:len(s)] + len(MON_FILTRE) - 1, s, c="tab:blue", lw=.8); a_sor.axhline(0, c="0.6", lw=.5)
    a_sor.set_title(f"sortie (max {s.max():.2f})", fontsize=8)
legende(fig); plt.tight_layout(rect=(0, .04, 1, 1)); plt.show()

### Exercice 3 — même filtre, autre canal

Le réseau voit **deux canaux** : `log` = ln(1+P)/ln(3001) et `lin` = P/3000.
Prédis : le filtre A (fronts) sur une **extinction** (≈ 10 W → 30–40 W → 10 W) donnera-t-il un signal
visible sur `lin` ? et sur `log` ? Même question pour le début d'un café.

**Mes prédictions :**

In [ ]:
for nom in ("extinction", "cafe_1"):
    montrer(nom, {"A sur log": [-1, 1]}, canal=0, titre_canal="ln(1+P)/ln(3001)")
    montrer(nom, {"A sur lin": [-1, 1]}, canal=1, titre_canal="P / 3000")

Un filtre du réseau a **un jeu de poids par canal** (forme : canaux × k) et additionne les deux :
il peut donc combiner « petit front en log » (extinction, pic) et « niveau exact en lin » (1400 W contre 2200 W).

## Étape 2 — Pourquoi ton perceptron a du mal

La première couche de ton perceptron [D50] a 50 neurones × 256 poids (2 canaux × 128 s).
Chaque poids est attaché à **une seconde précise de la fenêtre**.

**Prédiction** : si on dessine les poids d'un neurone le long des 128 secondes, à quoi ça ressemblera ?
Un motif court répété ? Un « gabarit » à un endroit précis ? Du bruit ?

**Mes prédictions :**

In [ ]:
ID_PERCEPTRON = "8912c2d8d3"      # le D50 du 13/10

m = np.load(os.path.join(RACINE, "banc", "resultats", "modeles", ID_PERCEPTRON + ".npz"))
W0 = m["W0"].reshape(m["W0"].shape[0], 2, -1)      # (neurones, canaux, secondes)
fort = np.argsort(-np.abs(m["W1"]).sum(0))[:8]      # les 8 neurones les plus utilisés par la sortie
fig, axes = plt.subplots(2, 4, figsize=(15, 5), sharex=True)
for ax, n in zip(axes.flat, fort):
    ax.plot(W0[n, 0], lw=.8, label="log"); ax.plot(W0[n, 1], lw=.8, label="lin")
    ax.axvline(127, c="r", ls=":", lw=.8); ax.set_title(f"neurone {n}", fontsize=9)
axes[0, 0].legend(fontsize=7); fig.supxlabel("seconde de la fenêtre (127 = maintenant)")
plt.tight_layout(); plt.show()
print("poids de la 1re couche :", m["W0"].size)

### À observer

Les poids se concentrent sur **les 20–30 dernières secondes** (un tiers de leur masse sur les 20 dernières,
contre moins d'un dixième sur les 20 premières), et chaque neurone a appris un gabarit **à une position
fixe**, souvent collé à la seconde « maintenant ». Le même motif 10 s plus tôt ou 10 s plus tard doit être
appris par un autre neurone : c'est ce qui coûte des poids, des exemples, et ce qui limite le plateau à ~91 %.

Un filtre de convolution de 7 secondes sur 2 canaux : **14 poids** (+ 1 biais), valables partout.
8 filtres : 120 paramètres, contre 12 800 pour cette couche dense.
Mais attention : après la convolution, il faut toujours quelque chose qui dise **quand** le motif est
apparu (« montée il y a 20 s, aucun creux depuis ») : c'est le rôle des couches suivantes.

## Étape 3 — Une seule couche apprise : qu'a-t-elle trouvé ?

Un « microscope », pas le réseau final. Dans `banc/banc_esp.ipynb`, cellule 1 :

```python
TACHE_OPTIONS = {"synth": 0.22, "equilibrage": "aucun"}
COUCHES = [{"type": "conv1d", "canaux": 8, "noyau": 7, "act": "relu"},
           {"type": "dense", "n": 32, "act": "relu"}]
```

(8 filtres de 7 s ; leurs 8 sorties de 122 s vont dans une couche dense de 32, puis la sortie à 12 classes.)

**Prédictions** : quels filtres apprendra-t-il ? Des fronts (comme A) ? des creux (comme C) ? des niveaux (comme B) ?
Sur quel canal ? Et la précision par fenêtre : mieux ou moins bien que le perceptron (91 %) ?

**Mes prédictions :**

In [ ]:
ID_CONV = None        # identifiant du journal du banc, après entraînement

if ID_CONV:
    m = np.load(os.path.join(RACINE, "banc", "resultats", "modeles", ID_CONV + ".npz"))
    Wc, bc = m["W0"], m["b0"]                       # (filtres, canaux, k), (filtres,)
    F, C, k = Wc.shape
    fig, axes = plt.subplots(2, (F + 1) // 2, figsize=(15, 5), sharey=True)
    for f, ax in enumerate(axes.flat[:F]):
        ax.bar(np.arange(k) - .2, Wc[f, 0], .4, label="log"); ax.bar(np.arange(k) + .2, Wc[f, 1], .4, label="lin")
        ax.axhline(0, c="0.5", lw=.5); ax.set_title(f"filtre {f}  (b = {bc[f]:.2f})", fontsize=9)
    axes.flat[0].legend(fontsize=7); fig.suptitle("Filtres appris (poids par seconde, par canal)")
    plt.tight_layout(); plt.show()

    # sorties des filtres (après ReLU) sur quelques événements
    for nom in ("cafe_1", "allumage", "grille_pain", "pic"):
        e = EV[nom]; t = np.arange(len(e["p"])) - e["debut"]
        cartes = np.array([np.maximum(sum(convolution(e["x"][c], Wc[f, c]) for c in range(C)) + bc[f], 0)
                           for f in range(F)])
        fig, (a1, a2) = plt.subplots(2, 1, figsize=(14, 4.5), sharex=True, height_ratios=[1, 2])
        a1.plot(t, e["p"], c="k", lw=.8); a1.set_yscale("symlog", linthresh=30); bandeau(a1, e["y"], t)
        a1.set_title(nom, fontsize=9)
        a2.imshow(cartes, aspect="auto", cmap="magma", interpolation="nearest",
                  extent=(t[k - 1] - .5, t[-1] + .5, F - .5, -.5))
        a2.set_ylabel("filtre"); a2.set_xlabel("secondes")
        legende(fig); plt.tight_layout(rect=(0, .05, 1, 1)); plt.show()
else:
    print("Entraîne d'abord le réseau de l'étape 3 dans le banc, puis renseigne ID_CONV.")

## Étape 4 — Empiler : ce qu'un neurone « voit »

Une 2e convolution ne regarde pas le signal, mais les **sorties de la 1re**. Un neurone de la couche 2
avec un noyau de 5 voit 5 sorties de la couche 1, qui voyaient chacune 7 s : il voit donc 7 + 4 = 11 s.
Avec un **max-pooling** de 2 entre les deux, chaque sortie de la couche 1 « compte double » :
il voit 7 + 1 + 4 × 2 = 16 s. C'est le **champ récepteur**.

**Exercice** : pour l'empilement ci-dessous, calcule à la main le champ récepteur après chaque couche,
puis vérifie avec la cellule suivante.

`conv k7 → maxpool 2 → conv k5 → maxpool 2 → conv k5 → maxpool 2`

**Le piège** : le max-pooling du banc regroupe les secondes par paires **depuis le début** de la fenêtre.
Si la longueur est impaire, la **dernière** seconde, la plus récente, celle qui compte pour la latence, est jetée.

**Mes calculs :**

In [ ]:
def champ_recepteur(longueur, couches):
    # longueur de sortie, champ récepteur et « pas » cumulé de chaque couche ; signale les secondes récentes perdues
    L, champ, pas = longueur, 1, 1
    print(f"{'couche':22s} {'sortie':>7s} {'voit (s)':>9s} {'pas (s)':>8s}")
    print(f"{'entrée':22s} {L:7d} {champ:9d} {pas:8d}")
    for c in couches:
        if c["type"] == "conv1d":
            k, s = c["noyau"], c.get("pas", 1)
            perdu = (L - k) % s
            L = (L - k) // s + 1
            champ += (k - 1) * pas
            pas *= s
            nom = f"conv k{k}" + (f" pas {s}" if s > 1 else "")
        elif c["type"] == "maxpool1d":
            t = c["taille"]
            perdu = L % t
            L = L // t
            champ += (t - 1) * pas
            pas *= t
            nom = f"maxpool {t}"
        else:
            print(f"{'dense ' + str(c['n']):22s}   voit toute la fenêtre"); continue
        alerte = f"   ⚠ {perdu} élément(s) le(s) plus récent(s) jeté(s)" if perdu else ""
        print(f"{nom:22s} {L:7d} {champ:9d} {pas:8d}{alerte}")

PILE = [{"type": "conv1d", "noyau": 7}, {"type": "maxpool1d", "taille": 2},
        {"type": "conv1d", "noyau": 5}, {"type": "maxpool1d", "taille": 2},
        {"type": "conv1d", "noyau": 5}, {"type": "maxpool1d", "taille": 2},
        {"type": "dense", "n": 32}]
for longueur in (128, 126):
    print(f"\nfenêtre de {longueur} s"); champ_recepteur(longueur, PILE)

## Étape 5 — Concevoir notre réseau à partir des motifs

On part de **ce qu'il faut reconnaître** et de **combien de temps ça dure** ; on en déduit les noyaux,
la profondeur, le pooling. Complète le tableau (durées : voir les traces ci-dessus et `spectrasense`).

| Motif | Durée typique | Couche qui doit le voir | Remarque |
|---|---|---|---|
| front de montée / descente | 1–2 s | | |
| pic de maintien | 1–4 s | | |
| creux du broyage | 1–3 s | | |
| oscillation de la régulation | ~5 s (période) | | |
| traîne de fin (~30 W) | 5–8 s | | |
| extinction (25–80 W) | ~16 s | | |
| palier de chauffe | 13–25 s | | |
| grille-pain : plat, sans creux | > 20 s | | |

Questions à trancher ensemble :
1. Taille des noyaux de la 1re couche : le plus petit motif utile fait combien ?
2. Champ récepteur total : de combien de secondes a-t-on besoin pour **décider** (grille-pain contre allumage) ?
3. Pooling ou convolution à pas de 2 ? Que perd-on en latence quand on réduit la résolution ?
4. Combien de filtres par couche, pour ~2 000 événements réels d'apprentissage ?
5. Et après les convolutions : que doit faire la partie dense ?